# 00 - Benchmark Model

AAI-540 Group 5. Setup, data prep, split, and encoding are reused from `train-model.ipynb` so the benchmark is evaluated on the exact same data. The benchmark is a majority-class model: the floor every real model must beat.

In [1]:
%pip install -q "sagemaker>=2.200,<3" xgboost

Note: you may need to restart the kernel to use updated packages.


## 1. Imports and AWS setup

In [2]:
import os
import json
import tarfile
import warnings
from pathlib import Path

import boto3
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sagemaker
from sagemaker import get_execution_role
from sagemaker.feature_store.feature_group import FeatureGroup
from sagemaker.session import Session
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
)
from sklearn.calibration import calibration_curve

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

region = boto3.Session().region_name
boto_session = boto3.Session(region_name=region)
sagemaker_client = boto_session.client(
    service_name="sagemaker",
    region_name=region
)
featurestore_runtime = boto_session.client(
    service_name="sagemaker-featurestore-runtime", region_name=region
)

feature_store_session = Session(
    boto_session=boto_session,
    sagemaker_client=sagemaker_client,
    sagemaker_featurestore_runtime_client=featurestore_runtime,
)


role = get_execution_role()
default_s3_bucket_name = feature_store_session.default_bucket()
prefix = "carepath-analytics/feature-store"

athena_s3_staging_dir = f"s3://{default_s3_bucket_name}/athena/staging"

print(f"AWS Region: {region}")
print(f"Execution Role ARN: {role}")
print(f"Feature Store S3 Prefix: s3://{default_s3_bucket_name}/{prefix}")
print(f"Athena Connection S3 Staging Dir: {athena_s3_staging_dir}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


/opt/conda/lib/python3.12/site-packages/sagemaker/__init__.py:86: SageMakerV2DeprecationWarning: You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation()
You are using the SageMaker Python SDK v2, which is on path of deprecation. v3 is the actively developed major version.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


AWS Region: us-east-1
Execution Role ARN: arn:aws:iam::135844594927:role/LabRole
Feature Store S3 Prefix: s3://sagemaker-us-east-1-135844594927/carepath-analytics/feature-store
Athena Connection S3 Staging Dir: s3://sagemaker-us-east-1-135844594927/athena/staging


# Data Engineering

- `diabetic_data.csv` - main encounter data
- `IDS_mapping.csv` - lookup values for admission type, discharge disposition, and admission source

## 2. Load both project data sources

In [3]:
data_candidates = [
    Path("../../data"),
    Path("../data"),
    Path("data"),
    Path("/home/sagemaker-user/diabetes-readmission-prediction/data"),
]

data_dir = next(
    (
        p for p in data_candidates
        if (p / "diabetic_data.csv").exists() and (p / "IDS_mapping.csv").exists()
    ),
    None,
)

if data_dir is None:
    raise FileNotFoundError(
        "Could not find diabetic_data.csv and IDS_mapping.csv. "
        "Update data_candidates with the correct project data folder."
    )

diabetes_path = data_dir / "diabetic_data.csv"
mapping_path = data_dir / "IDS_mapping.csv"

raw_df = pd.read_csv(diabetes_path)

print("Data folder:", data_dir)
print("Main dataset shape:", raw_df.shape)
raw_df.head(3)

Data folder: ../../data
Main dataset shape: (101766, 50)


,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,...,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,...,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,...,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,...,No,No,No,No,No,No,No,No,Yes,NO


## 3. Read `IDS_mapping.csv`

The lookup file contains three mapping sections in one CSV:

- `admission_type_id`
- `discharge_disposition_id`
- `admission_source_id`

The helper below reads each section and creates a dictionary.

In [4]:
def load_id_mappings(path):
    mappings = {}
    current_key = None

    with open(path, "r", encoding="utf-8-sig") as f:
        lines = [line.rstrip("\n\r") for line in f]

    valid_headers = {
        "admission_type_id",
        "discharge_disposition_id",
        "admission_source_id",
    }

    for line in lines:
        if not line.strip():
            continue

        parts = line.split(",", 1)
        first = parts[0].strip()

        if first in valid_headers:
            current_key = first
            mappings[current_key] = {}
            continue

        if current_key is None:
            continue

        if first == "":
            current_key = None
            continue

        if len(parts) < 2:
            continue

        try:
            key = int(first)
        except ValueError:
            continue

        description = parts[1].strip().strip('"').strip()

        if description.upper() in {
            "NULL",
            "NOT MAPPED",
            "UNKNOWN/INVALID",
            "NOT AVAILABLE",
        }:
            description = "Unknown"

        mappings[current_key][key] = description

    return mappings

id_mappings = load_id_mappings(mapping_path)

for name, mapping in id_mappings.items():
    print(name, "->", len(mapping), "mapped values")
    print(list(mapping.items())[:5])
    print()

admission_type_id -> 8 mapped values
[(1, 'Emergency'), (2, 'Urgent'), (3, 'Elective'), (4, 'Newborn'), (5, 'Unknown')]

discharge_disposition_id -> 30 mapped values
[(1, 'Discharged to home'), (2, 'Discharged/transferred to another short term hospital'), (3, 'Discharged/transferred to SNF'), (4, 'Discharged/transferred to ICF'), (5, 'Discharged/transferred to another type of inpatient care institution')]

admission_source_id -> 25 mapped values
[(1, 'Physician Referral'), (2, 'Clinic Referral'), (3, 'HMO Referral'), (4, 'Transfer from a hospital'), (5, 'Transfer from a Skilled Nursing Facility (SNF)')]



## 4. Store raw data in Amazon S3

Uses Amazon S3 as the main project storage.  
The original source files are uploaded to a `raw/` location before preprocessing.

In [5]:
session = sagemaker.Session()
bucket = session.default_bucket()

PROJECT_PREFIX = "carepath-analytics"
RAW_PREFIX = f"{PROJECT_PREFIX}/raw"

raw_main_s3_uri = session.upload_data(
    path=str(diabetes_path),
    bucket=bucket,
    key_prefix=RAW_PREFIX,
)

raw_mapping_s3_uri = session.upload_data(
    path=str(mapping_path),
    bucket=bucket,
    key_prefix=RAW_PREFIX,
)

print("Raw main dataset:", raw_main_s3_uri)
print("Raw ID mapping file:", raw_mapping_s3_uri)

Raw main dataset: s3://sagemaker-us-east-1-135844594927/carepath-analytics/raw/diabetic_data.csv
Raw ID mapping file: s3://sagemaker-us-east-1-135844594927/carepath-analytics/raw/IDS_mapping.csv


## 5. Clean and preprocess the data
- Convert placeholder values to consistent missing/unknown values
- Convert the original readmission target into a binary 30-day flag
- Translate coded hospital ID fields using `IDS_mapping.csv`
- Group high-cardinality diagnosis codes into broader clinical categories
- Remove duplicate encounters

In [6]:
df = raw_df.copy()

# Standardize missing placeholders
df.replace("?", "Unknown", inplace=True)
df.fillna("Unknown", inplace=True)

# Remove duplicate encounter records
df = df.drop_duplicates(subset=["encounter_id"]).reset_index(drop=True)

# Binary target: 1 = readmitted within 30 days, 0 = otherwise
df["readmitted_30d"] = (df["readmitted"] == "<30").astype(int)

# Map coded fields to readable descriptions using IDS_mapping.csv
df["admission_type"] = (
    pd.to_numeric(df["admission_type_id"], errors="coerce")
    .map(id_mappings["admission_type_id"])
    .fillna("Unknown")
)

df["discharge_disposition"] = (
    pd.to_numeric(df["discharge_disposition_id"], errors="coerce")
    .map(id_mappings["discharge_disposition_id"])
    .fillna("Unknown")
)

df["admission_source"] = (
    pd.to_numeric(df["admission_source_id"], errors="coerce")
    .map(id_mappings["admission_source_id"])
    .fillna("Unknown")
)

def categorize_icd9(code):
    code_str = str(code).strip().upper()

    if not code_str or code_str in {"UNKNOWN", "NAN", "NONE"}:
        return "Unknown"

    if code_str.startswith(("V", "E")):
        return "Other"

    try:
        value = int(code_str.split(".")[0][:3])
    except ValueError:
        return "Other"

    if (390 <= value <= 459) or value == 785:
        return "Circulatory"
    elif (460 <= value <= 519) or value == 786:
        return "Respiratory"
    elif (520 <= value <= 579) or value == 787:
        return "Digestive"
    elif value == 250:
        return "Diabetes"
    elif 800 <= value <= 999:
        return "Injury"
    elif 710 <= value <= 739:
        return "Musculoskeletal"
    elif (580 <= value <= 629) or value == 788:
        return "Genitourinary"
    elif 140 <= value <= 239:
        return "Neoplasms"
    else:
        return "Other"

for col in ["diag_1", "diag_2", "diag_3"]:
    df[f"{col}_group"] = df[col].apply(categorize_icd9)

print("Processed shape:", df.shape)
print("30-day readmission count:", int(df["readmitted_30d"].sum()))
print("30-day readmission rate:", round(df["readmitted_30d"].mean(), 4))

Processed shape: (101766, 57)
30-day readmission count: 11357
30-day readmission rate: 0.1116


## 6. Save processed data to S3

The processed dataset is stored separately from the raw files, which keeps the workflow reproducible.

In [7]:
PROCESSED_PREFIX = f"{PROJECT_PREFIX}/processed"

processed_dir = Path("artifacts")
processed_dir.mkdir(exist_ok=True)

processed_path = processed_dir / "diabetes_processed.csv"
df.to_csv(processed_path, index=False)

processed_s3_uri = session.upload_data(
    path=str(processed_path),
    bucket=bucket,
    key_prefix=PROCESSED_PREFIX,
)

print("Processed dataset:", processed_s3_uri)

Processed dataset: s3://sagemaker-us-east-1-135844594927/carepath-analytics/processed/diabetes_processed.csv


# Training Data

To reduce data leakage, the split is performed at the patient level so the same patient does not appear in multiple datasets.

## 7. Select features and target

In [8]:
categorical_cols = [
    "age",
    "admission_type",
    "discharge_disposition",
    "admission_source",
    "diag_1_group",
    "diag_2_group",
    "diag_3_group",
    "max_glu_serum",
    "A1Cresult",
    "change",
    "diabetesMed",
]

numeric_cols = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
]

feature_cols = categorical_cols + numeric_cols

# encounter_id and patient_nbr are kept only for tracking/splitting, not model inputs
X = df[feature_cols].copy()
y = df["readmitted_30d"].copy()
patient_ids = df["patient_nbr"].astype(str).copy()

# Sensitive attributes retained separately for the bias assessment
sensitive = df[["race", "gender", "age"]].copy()

print("Model feature count:", len(feature_cols))
print(feature_cols)

Model feature count: 19
['age', 'admission_type', 'discharge_disposition', 'admission_source', 'diag_1_group', 'diag_2_group', 'diag_3_group', 'max_glu_serum', 'A1Cresult', 'change', 'diabetesMed', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']


## 8. Create 70/15/15 patient-level train, validation, and test splits

In [9]:
patient_table = (
    pd.DataFrame({
        "patient_nbr": patient_ids,
        "target": y,
    })
    .groupby("patient_nbr", as_index=False)["target"]
    .max()
)

train_patients, temp_patients = train_test_split(
    patient_table,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=patient_table["target"],
)

validation_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_patients["target"],
)

train_ids = set(train_patients["patient_nbr"])
validation_ids = set(validation_patients["patient_nbr"])
test_ids = set(test_patients["patient_nbr"])

train_mask = patient_ids.isin(train_ids)
validation_mask = patient_ids.isin(validation_ids)
test_mask = patient_ids.isin(test_ids)

X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

X_validation = X.loc[validation_mask].copy()
y_validation = y.loc[validation_mask].copy()

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()

sensitive_test = sensitive.loc[test_mask].reset_index(drop=True)

print("Train:", X_train.shape, "positive rate:", round(y_train.mean(), 4))
print("Validation:", X_validation.shape, "positive rate:", round(y_validation.mean(), 4))
print("Test:", X_test.shape, "positive rate:", round(y_test.mean(), 4))

assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)

print("Patient leakage check: PASSED")

Train: (71178, 19) positive rate: 0.1116
Validation: (15310, 19) positive rate: 0.1127
Test: (15278, 19) positive rate: 0.1102
Patient leakage check: PASSED


# Feature Engineering

## 9. Encode and transform features

In [10]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore", sparse_output=True),
            categorical_cols,
        ),
        (
            "numeric",
            StandardScaler(),
            numeric_cols,
        ),
    ],
    remainder="drop",
)

X_train_t = preprocessor.fit_transform(X_train)
X_validation_t = preprocessor.transform(X_validation)
X_test_t = preprocessor.transform(X_test)

transformed_feature_names = preprocessor.get_feature_names_out().tolist()

print("Transformed training shape:", X_train_t.shape)
print("Transformed feature count:", len(transformed_feature_names))

Transformed training shape: (71178, 104)
Transformed feature count: 104


# Benchmark Model

## 10. Majority-class benchmark

Always predicts the most common class (not readmitted). Recall should be 0, which shows why accuracy alone is misleading on imbalanced data.

In [11]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

benchmark_model = DummyClassifier(strategy="most_frequent")
benchmark_model.fit(X_train_t, y_train)

def evaluate(model, X_t, y_true, name, split):
    prob = model.predict_proba(X_t)[:, 1]
    pred = model.predict(X_t)
    return {
        "model": name,
        "split": split,
        "accuracy": accuracy_score(y_true, pred),
        "pr_auc": average_precision_score(y_true, prob),
        "roc_auc": roc_auc_score(y_true, prob),
        "recall": recall_score(y_true, pred, zero_division=0),
        "precision": precision_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
    }

benchmark_results = pd.DataFrame([
    evaluate(benchmark_model, X_validation_t, y_validation, "Benchmark (majority class)", "validation"),
    evaluate(benchmark_model, X_test_t, y_test, "Benchmark (majority class)", "test"),
])

benchmark_results.round(4)

,model,split,accuracy,pr_auc,roc_auc,recall,precision,f1
0,Benchmark (majority class),validation,0.8873,0.1127,0.5,0.0,0.0,0.0
1,Benchmark (majority class),test,0.8898,0.1102,0.5,0.0,0.0,0.0


## 11. Confusion matrix (test)

In [12]:
bench_test_pred = benchmark_model.predict(X_test_t)
print(confusion_matrix(y_test, bench_test_pred))
print("Positive rate in test:", round(y_test.mean(), 4))

[[13594     0]
 [ 1684     0]]
Positive rate in test: 0.1102


## 12. Save benchmark metrics to S3

In [13]:
artifact_dir = Path("artifacts")
artifact_dir.mkdir(exist_ok=True)

benchmark_path = artifact_dir / "benchmark_metrics.json"
benchmark_results.to_json(benchmark_path, orient="records", indent=2)

benchmark_s3_uri = session.upload_data(
    path=str(benchmark_path),
    bucket=bucket,
    key_prefix=f"{PROJECT_PREFIX}/model-artifacts/benchmark",
)
print("Benchmark metrics:", benchmark_s3_uri)

Benchmark metrics: s3://sagemaker-us-east-1-135844594927/carepath-analytics/model-artifacts/benchmark/benchmark_metrics.json


## Takeaway

The benchmark scores ~0 recall and PR-AUC equal to the positive rate. Any model in `train-model.ipynb` must clearly beat these numbers on recall and PR-AUC to be worth deploying.